# Grid MiniYOLO loss

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/07-loss/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本節在 CPU 逐節執行並保存輸出；合成資料短訓練、L4 與部署紀錄見網站驗證頁。真實場景長訓練仍待安排。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.2.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 逐節範例預設 CPU；L4 訓練與部署的實測另見網站驗證頁。保留相同版本的 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
import math
import torch
from miniyolo.targets import build_targets
from miniyolo.losses import grid_loss


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    scene = {"boxes": torch.tensor([[8., 12., 24., 28.]]), "labels": torch.tensor([0])}
    target = build_targets([scene], 4, 64, 2)
    prediction = torch.zeros(1, 4, 4, 7, requires_grad=True)
    parts = grid_loss(prediction, target)
    expected_box = (.5**2 + .25**2 + .25**2 + .25**2) / 4
    assert abs(parts['box'].item() - expected_box) < 1e-7
    assert abs(parts['objectness'].item() - math.log(2)) < 1e-6
    assert abs(parts['classification'].item() - math.log(2)) < 1e-6
    assert abs(parts['total'].item() - (5*expected_box + 2*math.log(2))) < 1e-6
    parts['total'].backward()
    assert prediction.grad is not None and torch.isfinite(prediction.grad).all()
    assert prediction.grad[0, 0, 0, :4].abs().sum() == 0
    assert prediction.grad[0, 0, 0, 4] > 0  # gradient descent lowers background logit
    assert prediction.grad[0, 1, 1, 4] < 0  # raises positive objectness logit
    pos = target['positive']
    assert (prediction.grad[~pos][..., :4] == 0).all()
    assert (prediction.grad[~pos][..., 5:] == 0).all()
    assert torch.allclose(prediction.grad[..., 4][pos], torch.full((1,), -1/32))
    assert torch.allclose(prediction.grad[..., 4][~pos], torch.full((15,), 1/32))
    print({k: round(v.item(), 6) for k, v in parts.items()})
    print('positive/background objectness gradients',
          prediction.grad[0, 1, 1, 4].item(), prediction.grad[0, 0, 0, 4].item())
    empty = {"boxes": torch.empty(0, 4), "labels": torch.empty(0, dtype=torch.long)}
    empty_pred = torch.zeros(1, 4, 4, 7, requires_grad=True)
    empty_parts = grid_loss(empty_pred, build_targets([empty], 4, 64, 2))
    empty_parts['total'].backward()
    assert empty_parts['box'].item() == empty_parts['classification'].item() == 0
    assert torch.isfinite(empty_pred.grad).all()
    assert abs(empty_parts['objectness'].item() - math.log(2)) < 1e-6
    assert abs(empty_parts['total'].item() - math.log(2)) < 1e-6
    assert (empty_pred.grad[..., :4] == 0).all()
    assert (empty_pred.grad[..., 5:] == 0).all()
    assert torch.allclose(empty_pred.grad[..., 4], torch.full((1,4,4),1/32))
    print('empty image: finite backward, box/class=0')


if __name__ == '__main__':
    main()


{'total': 1.933169, 'box': 0.109375, 'objectness': 0.693147, 'classification': 0.693147}
positive/background objectness gradients -0.03125 0.03125
empty image: finite backward, box/class=0
